# MIDA507 -- Notebook Session 04
## Metadonnees DCAT et Catalogage Documentaire

| | |
|---|---|
| **Session** | S04 -- Metadonnees DCAT et catalogage |
| **Prerequis** | Notebooks S01-S03 |
| **Duree** | 70 a 90 minutes |

### Objectifs
1. Comprendre le standard DCAT (Data Catalog Vocabulary) du W3C
2. Distinguer les proprietes obligatoires, recommandees et optionnelles DCAT-AP
3. Construire une fiche DCAT complete pour notre jeu de bibliotheque UNG
4. Generer la notice au format JSON-LD et la valider
5. Creer une page HTML de notice lisible pour les usagers

### Livrables : `MIDA507_S04_fiche_dcat.json` + `MIDA507_S04_notice.html`


---
## PARTIE 1 -- Configuration


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as mpatches, seaborn as sns
import json, os, random, hashlib
from datetime import datetime, timedelta
import warnings; warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
PAL = {"ter1":"#3E1500","ter2":"#BF360C","ter3":"#E64A19",
       "olive":"#558B2F","bleu":"#1565C0","gris":"#546E7A","clair":"#FBE9E7"}
def regen(n=5000):
    random.seed(42); np.random.seed(42)
    g  = ["These et memoire","Manuel universitaire","Revue scientifique",
          "Rapport de recherche","Ouvrage de reference","Document administratif"]
    fi = ["Droit","Sciences eco.","Lettres modernes","Histoire",
          "Geographie","Medecine","Agronomie","Informatique"]
    ni = ["Licence 1","Licence 2","Licence 3","Master 1","Master 2","Doctorat"]
    re = ["Adamaoua","Centre","Est","Extreme-Nord","Littoral","Nord","Ouest","Sud"]
    la = ["Francais","Anglais","Bilingue FR/EN","Arabe","Autres lang. africaines"]
    li = ["fra","eng","fra+eng","ara","mul"]
    d0 = datetime(2018,1,1)
    df = pd.DataFrame({
        "ark_id":[f"ark:/67375/UNG-{str(i+1).zfill(6)}" for i in range(n)],
        "cote":[f"UNG-{str(i+1).zfill(5)}" for i in range(n)],
        "genre_document":np.random.choice(g,n,p=[0.28,0.30,0.15,0.10,0.10,0.07]),
        "date_emprunt":[(d0+timedelta(days=random.randint(0,365*5))).strftime("%Y-%m-%d") for _ in range(n)],
        "duree_jours":np.random.choice([3,7,14,21,30],n,p=[0.1,0.3,0.3,0.2,0.1]),
        "code_usager_anon":[f"USR-{hex(random.randint(0,0xFFFFFF))[2:].upper().zfill(6)}" for _ in range(n)],
        "filiere":np.random.choice(fi,n),
        "niveau":np.random.choice(ni,n),
        "region_origine":np.random.choice(re,n),
        "langue_document":np.random.choice(la,n,p=[0.62,0.22,0.10,0.04,0.02]),
        "langue_iso639":np.random.choice(li,n,p=[0.62,0.22,0.10,0.04,0.02]),
    })
    df["annee"]=pd.to_datetime(df["date_emprunt"]).dt.year
    return df
CSV = "MIDA507_S02_bibliotheque_ung_enrichie.csv"
if os.path.exists(CSV):
    df = pd.read_csv(CSV); print(f"Jeu S02 charge : {len(df):,} lignes")
else:
    df = regen(); print(f"Jeu S02 regenere : {len(df):,} lignes")
print(f"Colonnes : {list(df.columns)}")


---
## PARTIE 2 -- Le Standard DCAT : Vocabulaire des Catalogues de Donnees

> DCAT (Data Catalog Vocabulary) est le standard du W3C pour decrire les jeux de donnees.
> Il est utilise par data.gouv.bj, le portail europeen data.europa.eu et tous les portails CKAN modernes.
> En adoptant DCAT, notre jeu de donnees devient automatiquement interoperable avec des milliers de portails.


In [ ]:
from IPython.display import display, HTML

# Structure DCAT : 3 niveaux
DCAT_STRUCTURE = {
    "Catalogue (dcat:Catalog)": {
        "role": "Conteneur de jeux de donnees -- le portail lui-meme",
        "exemple": "Le portail CKAN de la BU-UNG",
        "proprietes": ["dct:title","dct:description","dcat:dataset","dct:publisher","dcat:themeTaxonomy"],
    },
    "Jeu de donnees (dcat:Dataset)": {
        "role": "Un jeu de donnees publie -- notre unite principale",
        "exemple": "Emprunts BU-UNG 2018-2023",
        "proprietes": ["dct:identifier","dct:title","dct:description","dcat:keyword",
                       "dct:issued","dct:modified","dct:license","dct:publisher",
                       "dct:spatial","dct:temporal","dcat:distribution"],
    },
    "Distribution (dcat:Distribution)": {
        "role": "Fichier ou API specifique permettant d'acceder au jeu",
        "exemple": "Fichier CSV UTF-8 ou API JSON",
        "proprietes": ["dcat:downloadURL","dcat:mediaType","dct:format",
                       "dct:byteSize","dcat:accessURL"],
    }
}

html = "<div style='font-family:Arial'>"
cols = ["#3E1500","#BF360C","#1565C0"]
for (niveau, info), col in zip(DCAT_STRUCTURE.items(), cols):
    html += f"""<div style="border-left:4px solid {col};padding:12px 16px;margin:10px 0;background:#FAFAFA">
      <h4 style="color:{col};margin:0 0 6px 0">{niveau}</h4>
      <p style="margin:4px 0;font-size:12px"><b>Role :</b> {info["role"]}</p>
      <p style="margin:4px 0;font-size:12px"><b>Exemple :</b> <i>{info["exemple"]}</i></p>
      <p style="margin:4px 0;font-size:12px"><b>Proprietes cles :</b> {', '.join(f'<code>{p}</code>' for p in info["proprietes"])}</p>
    </div>"""
html += "</div>"
display(HTML(html))
print("\nPrincipe DCAT : chaque jeu de donnees a une fiche (Dataset)")
print("qui peut pointer vers plusieurs fichiers (Distributions : CSV, JSON, API...)")


---
## PARTIE 3 -- Construire la Fiche DCAT de notre Jeu de Bibliotheque

> On complete maintenant les 14 champs DCAT obligatoires et recommandes
> pour transformer notre jeu en jeu de donnees professionnellement documente.


In [ ]:
def creer_fiche_dcat(df):
    """Genere une fiche DCAT complete pour notre jeu de bibliotheque UNG."""
    from datetime import date

    # Calculs automatiques depuis le DataFrame
    genres_uniques  = sorted(df["genre_document"].unique().tolist())
    langues_uniques = sorted(df["langue_iso639"].dropna().unique().tolist())
    regions_uniques = sorted(df["region_origine"].unique().tolist())
    date_min = df["date_emprunt"].min()
    date_max = df["date_emprunt"].max()
    nb_lignes = len(df)

    fiche = {
        "@context": "https://www.w3.org/ns/dcat#",
        "@type": "dcat:Dataset",

        # F -- FINDABLE
        "dct:identifier": "ark:/67375/UNG-BIBLIO-EMPRUNTS-2018-2023",
        "dct:title": {
            "fr": "Emprunts documentaires -- Bibliotheque Centrale Universite de Ngaoundere 2018-2023",
            "en": "Document Loans -- Central Library University of Ngaoundere 2018-2023"
        },
        "dct:description": {
            "fr": (f"Jeu de donnees tabulaires enregistrant {nb_lignes:,} emprunts documentaires "
                   f"effectues a la Bibliotheque Centrale de l'Universite de Ngaoundere (UNG) "
                   f"entre {date_min} et {date_max}. "
                   f"Couvre {len(genres_uniques)} genres documentaires, "
                   f"{len(langues_uniques)} langues (codes ISO 639) et "
                   f"les 10 regions du Cameroun."),
        },
        "dcat:keyword": [
            "bibliotheque universitaire","emprunt documentaire","Cameroun","Adamaoua",
            "sciences de l'information","data stewardship","education superieure",
            "MIDA507","Universite de Ngaoundere"
        ],
        "dcat:theme": [
            "http://publications.europa.eu/resource/authority/data-theme/EDUC",
            "http://publications.europa.eu/resource/authority/data-theme/SOCI"
        ],
        "dct:language": langues_uniques,

        # A -- ACCESSIBLE
        "dcat:landingPage": "https://ckan.demo.ung.cm/dataset/ung-biblio-emprunts-2018-2023",
        "dct:accessRights": "http://publications.europa.eu/resource/authority/access-right/PUBLIC",
        "dct:conformsTo": "https://www.w3.org/TR/vocab-dcat-2/",

        # I -- INTEROPERABLE
        "dct:accrualPeriodicity": "http://purl.org/cld/freq/annual",
        "dct:spatial": {
            "@type": "dct:Location",
            "locn:geometry": {"@type": "geo:wktLiteral",
                              "@value": "POINT (13.5780 7.3667)"},
            "skos:prefLabel": "Ngaoundere, Region de l'Adamaoua, Cameroun",
            "ISO_3166": "CM"
        },
        "dct:temporal": {
            "@type": "dct:PeriodOfTime",
            "dcat:startDate": date_min,
            "dcat:endDate": date_max
        },

        # R -- REUSABLE
        "dct:license": "https://creativecommons.org/licenses/by/4.0/",
        "dct:rights": "Les donnees sont publiees sous licence CC-BY 4.0. Toute reutilisation doit citer : Universite de Ngaoundere -- Bibliotheque Centrale, 2024.",
        "dct:publisher": {
            "@type": "org:Organization",
            "foaf:name": "Bibliotheque Centrale -- Universite de Ngaoundere",
            "foaf:mbox": "data-steward@univ-ngaoundere.cm",
            "foaf:page": "https://www.univ-ngaoundere.cm/bibliotheque"
        },
        "dct:creator": {
            "@type": "foaf:Person",
            "foaf:name": "Data Steward IDA -- MIDA507",
            "foaf:mbox": "data-steward@univ-ngaoundere.cm"
        },
        "dct:issued": date.today().isoformat(),
        "dct:modified": date.today().isoformat(),
        "dct:provenance": {
            "rdfs:label": "Donnees exportees depuis le SIGB PMB. 4 transformations documentees dans le journal de lineage S02. Codes usagers pseudonymises (hachage MD5)."
        },

        # Statistiques embarquees (enrichissement)
        "dcat:qualifiedRelation": {
            "statistiques": {
                "nb_enregistrements": nb_lignes,
                "genres_documentaires": genres_uniques,
                "langues_iso639": langues_uniques,
                "regions_cameroun": regions_uniques,
                "periode": f"{date_min} / {date_max}",
                "score_fair_avant_s04": 18,
                "score_fair_cible": 87
            }
        },

        # Distributions
        "dcat:distribution": [
            {
                "@type": "dcat:Distribution",
                "dct:title": "Fichier CSV UTF-8 -- Version complete",
                "dcat:downloadURL": "https://ckan.demo.ung.cm/dataset/ung-biblio/ung_emprunts_2018_2023.csv",
                "dcat:mediaType": "text/csv",
                "dct:format": "CSV",
                "dct:description": f"Fichier plat CSV UTF-8 avec BOM, {nb_lignes:,} lignes, {len(df.columns)} colonnes.",
                "dcat:byteSize": nb_lignes * len(df.columns) * 15
            },
            {
                "@type": "dcat:Distribution",
                "dct:title": "API JSON -- CKAN Resource API",
                "dcat:accessURL": "https://ckan.demo.ung.cm/api/3/action/datastore_search?resource_id=ung-emprunts",
                "dcat:mediaType": "application/json",
                "dct:format": "API REST JSON",
                "dct:description": "API CKAN permettant la requete filtree par genre, region, annee et langue."
            }
        ]
    }
    return fiche

fiche = creer_fiche_dcat(df)

# Affichage des champs principaux
print("FICHE DCAT -- Bibliotheque UNG")
print("="*55)
for champ in ["dct:identifier","dct:title","dcat:keyword","dct:license","dct:issued"]:
    val = fiche.get(champ,"")
    if isinstance(val,dict):
        val = val.get("fr",str(val))[:80]
    elif isinstance(val,list):
        val = ", ".join(str(v) for v in val[:3])+"..." if len(val)>3 else ", ".join(str(v) for v in val)
    print(f"  {champ:<25} : {val}")
print(f"  Distributions              : {len(fiche['dcat:distribution'])} (CSV + API JSON)")
print(f"  Mots-cles                  : {len(fiche['dcat:keyword'])} termes")


In [ ]:
# Sauvegarde JSON-LD
nom_json = "MIDA507_S04_fiche_dcat.json"
with open(nom_json,"w",encoding="utf-8") as f:
    json.dump(fiche, f, ensure_ascii=False, indent=2)
print(f"Fiche DCAT sauvegardee : {nom_json}")
print(f"Taille : {os.path.getsize(nom_json)} octets")
print()

# Evaluation de la completude DCAT
champs_obligatoires = ["dct:identifier","dct:title","dct:description",
                        "dcat:keyword","dct:publisher","dct:license","dcat:distribution"]
champs_recommandes  = ["dct:spatial","dct:temporal","dct:issued","dct:modified",
                        "dct:language","dcat:theme","dcat:landingPage"]
print("COMPLETUDE DCAT-AP :")
print("  Champs obligatoires :")
for c in champs_obligatoires:
    present = c in fiche
    print(f"    {'OK' if present else 'MANQUANT'} {c}")
print("  Champs recommandes :")
for c in champs_recommandes:
    present = c in fiche
    print(f"    {'OK' if present else 'MANQUANT'} {c}")
score_ob  = sum(1 for c in champs_obligatoires if c in fiche)
score_rec = sum(1 for c in champs_recommandes  if c in fiche)
print(f"\n  Score obligatoires  : {score_ob}/{len(champs_obligatoires)}")
print(f"  Score recommandes   : {score_rec}/{len(champs_recommandes)}")
print(f"  Score DCAT global   : {score_ob+score_rec}/{len(champs_obligatoires)+len(champs_recommandes)}")


---
## PARTIE 4 -- Generer la Notice HTML de Consultation

> La notice HTML est la page que voit l'usager quand il arrive sur le portail.
> Elle doit etre claire, complete et lisible sur mobile (81% d'usage mobile en Afrique).


In [ ]:
def generer_notice_html(fiche, df):
    """Genere une notice HTML responsive depuis la fiche DCAT."""
    titre = fiche["dct:title"]["fr"] if isinstance(fiche["dct:title"],dict) else fiche["dct:title"]
    desc  = fiche["dct:description"]["fr"] if isinstance(fiche["dct:description"],dict) else ""
    mots_cles = ", ".join(fiche.get("dcat:keyword",[]))
    licence_url = fiche.get("dct:license","")
    editeur = fiche.get("dct:publisher",{}).get("foaf:name","")
    date_pub = fiche.get("dct:issued","")
    ark = fiche.get("dct:identifier","")
    nb_lignes = len(df)
    distributions = fiche.get("dcat:distribution",[])

    distrib_html = ""
    for d in distributions:
        fmt  = d.get("dct:format","")
        url  = d.get("dcat:downloadURL") or d.get("dcat:accessURL","")
        dt   = d.get("dct:title","")
        distrib_html += f"""
      <div style="border:1px solid #ddd;border-radius:6px;padding:12px;margin:8px 0;background:#f9f9f9">
        <strong>{fmt}</strong> -- {dt}<br>
        <a href="{url}" style="color:#1565C0;font-size:12px">{url}</a>
      </div>"""

    html = f"""<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{titre}</title>
<style>
  body{{font-family:Arial,sans-serif;max-width:800px;margin:0 auto;padding:16px;color:#333}}
  h1{{color:#3E1500;font-size:20px;border-bottom:3px solid #E64A19;padding-bottom:8px}}
  .meta{{background:#FBE9E7;border-radius:8px;padding:14px;margin:12px 0}}
  .meta table{{width:100%;border-collapse:collapse;font-size:13px}}
  .meta td{{padding:5px 8px;vertical-align:top}}
  .meta td:first-child{{font-weight:bold;color:#BF360C;width:180px}}
  .badge{{display:inline-block;background:#3E1500;color:white;padding:3px 8px;
          border-radius:12px;font-size:11px;margin:3px 2px}}
  h2{{color:#1565C0;font-size:15px;margin-top:20px}}
  .stat{{display:inline-block;background:#E3F2FD;border-radius:6px;
         padding:8px 14px;margin:4px;text-align:center;min-width:110px}}
  .stat strong{{display:block;font-size:20px;color:#1565C0}}
</style>
</head>
<body>
<h1>{titre}</h1>

<div class="meta">
<table>
<tr><td>Identifiant</td><td><code>{ark}</code></td></tr>
<tr><td>Editeur</td><td>{editeur}</td></tr>
<tr><td>Date de publication</td><td>{date_pub}</td></tr>
<tr><td>Licence</td><td><a href="{licence_url}" style="color:#1565C0">CC-BY 4.0</a></td></tr>
<tr><td>Mots-cles</td><td>{" ".join(f'<span class="badge">{m}</span>' for m in fiche.get("dcat:keyword",[]))}</td></tr>
</table>
</div>

<h2>Description</h2>
<p style="font-size:13px;line-height:1.6">{desc}</p>

<h2>Statistiques du jeu</h2>
<div>
  <div class="stat"><strong>{nb_lignes:,}</strong>enregistrements</div>
  <div class="stat"><strong>{len(df.columns)}</strong>colonnes</div>
  <div class="stat"><strong>2018-2023</strong>periode</div>
  <div class="stat"><strong>10</strong>regions CM</div>
</div>

<h2>Telechargements</h2>
{distrib_html}

<hr style="margin:20px 0;border-color:#eee">
<p style="font-size:11px;color:#999">
  Fiche DCAT-AP generee par MIDA507 -- Master MIDA -- Universite de Douala. {date_pub}.
</p>
</body>
</html>"""
    return html

html_notice = generer_notice_html(fiche, df)
nom_html = "MIDA507_S04_notice.html"
with open(nom_html,"w",encoding="utf-8") as f:
    f.write(html_notice)
print(f"Notice HTML generee : {nom_html} ({os.path.getsize(nom_html)} octets)")
print()
print("Affichage dans Colab :")
from IPython.display import display, HTML
display(HTML(html_notice))


---
### EXERCICE -- Completez une fiche DCAT pour un jeu de votre institution


In [ ]:
# ============================================================
# EXERCICE -- Creez la fiche DCAT de votre institution
# ============================================================
# Remplissez les champs ci-dessous avec votre jeu de donnees reel.

MON_TITRE       = "[Titre descriptif : institution + type + periode]" # <- changez
MON_ARK         = "[ark:/NAAN/institution-type-annee]"                # <- changez
MON_EDITEUR     = "[Nom de votre institution]"                        # <- changez
MA_LICENCE      = "https://creativecommons.org/licenses/by/4.0/"     # <- CC-BY recommande
MES_MOTS_CLES   = ["[mot-cle 1]","[mot-cle 2]","[mot-cle 3]"]       # <- au moins 5
MA_DESCRIPTION  = "[Description complete : sujet, periode, couverture, methode]" # <- changez

MA_FICHE = {
    "dct:identifier":  MON_ARK,
    "dct:title":       {"fr": MON_TITRE},
    "dct:description": {"fr": MA_DESCRIPTION},
    "dcat:keyword":    MES_MOTS_CLES,
    "dct:publisher":   {"foaf:name": MON_EDITEUR},
    "dct:license":     MA_LICENCE,
    "dct:issued":      datetime.now().strftime("%Y-%m-%d"),
}

# Verification des champs obligatoires
OBLIGATOIRES = ["dct:identifier","dct:title","dct:description",
                "dcat:keyword","dct:publisher","dct:license"]
print("VERIFICATION DCAT OBLIGATOIRES :")
tous_ok = True
for c in OBLIGATOIRES:
    val = MA_FICHE.get(c)
    if isinstance(val,dict): val = list(val.values())[0] if val else None
    ok = bool(val) and "[" not in str(val)
    print(f"  {'OK' if ok else 'MANQUANT/MODIFIER'} {c:<30} : {str(val)[:50] if val else 'VIDE'}")
    if not ok: tous_ok = False

if tous_ok:
    with open("MIDA507_S04_ma_fiche_dcat.json","w",encoding="utf-8") as f:
        json.dump(MA_FICHE,f,ensure_ascii=False,indent=2)
    print("\nFiche sauvegardee : MIDA507_S04_ma_fiche_dcat.json")
else:
    print("\n[A COMPLETER] Modifiez les champs entre crochets ci-dessus.")


---
## Bilan S04

| Competence | Statut |
|---|---|
| Comprendre la structure DCAT (Catalog / Dataset / Distribution) | OK |
| Renseigner les 7 champs obligatoires DCAT-AP | OK |
| Generer une fiche DCAT JSON-LD complete | OK |
| Creer une notice HTML responsive pour les usagers | OK |
| Rediger ma propre fiche DCAT | A completer |

**Lien S05 :** Le jeu enrichi sera soumis a un audit de qualite DAMA complet (6 dimensions).

*Notebook MIDA507 S04 -- Master MIDA -- Universite de Douala*
